# Agentic Sales Pipeline

In [58]:
import sys
import os
import yaml
from crewai import Agent, Task, Crew

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you acan import your config
from config import api_key, serper_api_key, trello_api_key, trello_api_token, trello_board_id

import os

os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["OPENAI_API_KEY"] = api_key
os.environ["SERPER_API_KEY"] = serper_api_key

In [59]:
# Define file paths for YAML configurations
files = {
    'lead_agents': 'config3/lead_qualification_agents.yaml',
    'lead_tasks': 'config3/lead_qualification_tasks.yaml',
    'email_agents': 'config3/email_engagement_agents.yaml',
    'email_tasks': 'config3/email_engagement_tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
lead_agents_config = configs['lead_agents']
lead_tasks_config = configs['lead_tasks']
email_agents_config = configs['email_agents']
email_tasks_config = configs['email_tasks']

## Create Pydantic Models for Structured Output

In [60]:
from pydantic import BaseModel, Field
from typing import Dict, Optional, List, Set, Tuple

class LeadPersonalInfo(BaseModel):
    name: str = Field(..., description="The full name of the lead.")
    job_title: str = Field(..., description="The job title of the lead.")
    role_relevance: int = Field(..., ge=0, le=10, description="A score representing how relevant the lead's role is to the decision-making process (0-10).")
    professional_background: Optional[str] = Field(..., description="A brief description of the lead's professional background.")

class CompanyInfo(BaseModel):
    company_name: str = Field(..., description="The name of the company the lead works for.")
    industry: str = Field(..., description="The industry in which the company operates.")
    company_size: int = Field(..., description="The size of the company in terms of employee count.")
    revenue: Optional[float] = Field(None, description="The annual revenue of the company, if available.")
    market_presence: int = Field(..., ge=0, le=10, description="A score representing the company's market presence (0-10).")

class LeadScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="The final score assigned to the lead (0-100).")
    scoring_criteria: List[str] = Field(..., description="The criteria used to determine the lead's score.")
    validation_notes: Optional[str] = Field(None, description="Any notes regarding the validation of the lead score.")

class LeadScoringResult(BaseModel):
    personal_info: LeadPersonalInfo = Field(..., description="Personal information about the lead.")
    company_info: CompanyInfo = Field(..., description="Information about the lead's company.")
    lead_score: LeadScore = Field(..., description="The calculated score and related information for the lead.")

## Importing Tools

In [61]:
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

In [62]:
# Creating Agents
lead_data_agent = Agent(
  config=lead_agents_config['lead_data_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

cultural_fit_agent = Agent(
  config=lead_agents_config['cultural_fit_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

scoring_validation_agent = Agent(
  config=lead_agents_config['scoring_validation_agent'],
  tools=[SerperDevTool(), ScrapeWebsiteTool()]
)

# Creating Tasks
lead_data_task = Task(
  config=lead_tasks_config['lead_data_collection'],
  agent=lead_data_agent
)

cultural_fit_task = Task(
  config=lead_tasks_config['cultural_fit_analysis'],
  agent=cultural_fit_agent
)

scoring_validation_task = Task(
  config=lead_tasks_config['lead_scoring_and_validation'],
  agent=scoring_validation_agent,
  context=[lead_data_task, cultural_fit_task],
  output_pydantic=LeadScoringResult
)

# Creating Crew
lead_scoring_crew = Crew(
  agents=[
    lead_data_agent,
    cultural_fit_agent,
    scoring_validation_agent
  ],
  tasks=[
    lead_data_task,
    cultural_fit_task,
    scoring_validation_task
  ],
  verbose=True
)

## Email Engagement Crew

In [63]:
# Creating Agents
email_content_specialist = Agent(
  config=email_agents_config['email_content_specialist']
)

engagement_strategist = Agent(
  config=email_agents_config['engagement_strategist']
)

# Creating Tasks
email_drafting = Task(
  config=email_tasks_config['email_drafting'],
  agent=email_content_specialist
)

engagement_optimization = Task(
  config=email_tasks_config['engagement_optimization'],
  agent=engagement_strategist
)

# Creating Crew
email_writing_crew = Crew(
  agents=[
    email_content_specialist,
    engagement_strategist
  ],
  tasks=[
    email_drafting,
    engagement_optimization
  ],
  verbose=True
)

## Creating Complete Sales Flow

In [64]:
from crewai import Flow
from crewai.flow.flow import listen, start

class SalesPipeline(Flow):
    @start()
    def fetch_leads(self):
        # Pull our leads from the database
        leads = [
            {
                "lead_data": {
                    "name": "João Moura",
                    "job_title": "Director of Engineering",
                    "company": "Clearbit",
                    "email": "joao@clearbit.com",
                    "use_case": "Using AI Agent to do better data enrichment."
                },
            },
        ]
        return leads

    @listen(fetch_leads)
    def score_leads(self, leads):
        scores = lead_scoring_crew.kickoff_for_each(leads)
        self.state["score_crews_results"] = scores
        return scores

    @listen(score_leads)
    def store_leads_score(self, scores):
        # Here we would store the scores in the database
        return scores

    @listen(score_leads)
    def filter_leads(self, scores):
        return [score for score in scores if score['lead_score'].score > 70]

    @listen(filter_leads)
    def write_email(self, leads):
        scored_leads = [lead.to_dict() for lead in leads]
        emails = email_writing_crew.kickoff_for_each(scored_leads)
        return emails

    @listen(write_email)
    def send_email(self, emails):
        # Here we would send the emails to the leads
        return emails

## Plotting the Flow

In [65]:
flow = SalesPipeline()
flow.plot()

╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: SalesPipeline                                                                                            │
│  ID: f9b9d917-9c02-4c0c-9afc-dd887fc7b544                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Plot saved as crewai_flow.html


In [66]:
from IPython.display import IFrame

IFrame(src='./crewai_flow.html', width='150%', height=600)

## Flow Kickoff

In [67]:
emails = await flow.kickoff_async()

 Flow started with ID: f9b9d917-9c02-4c0c-9afc-dd887fc7b544

Output()

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 15ba18f2-e7bc-48d7-b64e-df1912574f25                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Data Specialist                                                                                    │
│                                                                                                                 │
│  Task: Collect and analyze the following information about the lead:                                            │
│  - Personal Information:                                                                                        │
│    - Name: Obtain the full name of the lead.                                                                    │
│    - Job Title: Determine the lead's current job title.                                                         │
│    - Role Relevance: Assess how relevant the lead's role is to the decision-making process on a scale from 0    │
│  to 10.                                                                                                         │
│    - Professional Background: Optionally, gather a brief description of the lead's professional background.     │
│                                                                                                                 │
│  - Company Information:                                                                                         │
│    - Company Name: Identify the name of the company the lead works for.                                         │
│    - Industry: Determine the industry in which the company operates.                                            │
│    - Company Size: Estimate the size of the company in terms of employee count.                                 │
│    - Revenue: If available, collect information on the annual revenue of the company.                           │
│    - Market Presence: Evaluate the company's market presence on a scale from 0 to 10.                           │
│                                                                                                                 │
│  - Our Company and Product:                                                                                     │
│    - Company Name: CrewAI                                                                                       │
│    - Product: Multi-Agent Orchestration Platform                                                                │
│    - ICP: Enterprise companies looking into Agentic automation.                                                 │
│    - Pitch: We are a platform that allows you to orchestrate AI Agents for automations to any vertical.         │
│                                                                                                                 │
│  -Lead Data:                                                                                                    │
│    {'name': 'João Moura', 'job_title': 'Director of Engineering', 'company': 'Clearbit', 'email':               │
│  'joao@clearbit.com', 'use_case': 'Using AI Agent to do better data enrichment.'}                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Engagement Optimization Specialist                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Unlock Automation Potential with CrewAI                                                               │
│                                                                                                                 │
│  Hi there,                                                                                                      │
│                                                                                                                 │
│  Are you ready to revolutionize your automation processes? CrewAI is the ultimate platform for orchestrating    │
│  AI Agents tailored for enterprise companies seeking efficient Agentic automation solutions.                    │
│                                                                                                                 │
│  Maximize your operational efficiency and schedule a personalized demo today to witness the power of            │
│  multi-agent orchestration in action. Don't miss out on the opportunity to streamline your operations           │
│  effortlessly.                                                                                                  │
│                                                                                                                 │
│  Schedule your demo now and embark on a journey towards unparalleled automation excellence with CrewAI.         │
│                                                                                                                 │
│  Best,                                                                                                          │
│                                                                                                                 │
│  CrewAI Team                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e62eb6ca-d716-4247-a011-3a6207e8b7ab                                                                     │
│  Agent: Engagement Optimization Specialist                                                                      │
│                                                                                                                 │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a0c9aa86-fe4b-4e84-b927-e9ebcb39e0e5                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: Subject: Unlock Automation Potential with CrewAI                                                 │
│                                                                                                                 │
│  Hi there,                                                                                                      │
│                                                                                                                 │
│  Are you ready to revolutionize your automation processes? CrewAI is the ultimate platform for orchestrating    │
│  AI Agents tailored for enterprise companies seeking efficient Agentic automation solutions.                    │
│                                                                                                                 │
│  Maximize your operational efficiency and schedule a personalized demo today to witness the power of            │
│  multi-agent orchestration in action. Don't miss out on the opportunity to streamline your operations           │
│  effortlessly.                                                                                                  │
│                                                                                                                 │
│  Schedule your demo now and embark on a journey towards unparalleled automation excellence with CrewAI.         │
│                                                                                                                 │
│  Best,                                                                                                          │
│                                                                                                                 │
│  CrewAI Team                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────────── Flow Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: SalesPipeline                                                                                            │
│  ID: f9b9d917-9c02-4c0c-9afc-dd887fc7b544                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Usage Metrics and Costs

Let’s see how much it would cost each time if this crew runs at sc

In [68]:
import pandas as pd

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([flow.state["score_crews_results"][0].token_usage.model_dump()])

# Calculate total costs
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Display the DataFrame
df_usage_metrics

Total costs: $0.0049


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,32900,31786,0,1114,13


In [69]:
print(flow.state['score_crews_results'][0].raw)

{
  "personal_info": {
    "name": "João Moura",
    "job_title": "Director of Engineering",
    "role_relevance": 8,
    "professional_background": "Experienced engineering leader with a background in technology and leadership roles."
  },
  "company_info": {
    "company_name": "Clearbit",
    "industry": "Internet Software & Services",
    "company_size": 120,
    "revenue": null,
    "market_presence": null
  },
  "lead_score": {
    "score": 85,
    "scoring_criteria": [
      "Role Relevance: 8/10",
      "Company Size: 120 employees",
      "Market Presence: Not available",
      "Cultural Fit: 8/10"
    ],
    "validation_notes": "João Moura demonstrates strong role relevance with a background suitable for the Director of Engineering position. The lead score reflects the alignment of his experience with the company's needs."
  }
}


In [70]:
import pandas as pd

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([emails[0].token_usage.model_dump()])

# Calculate total costs
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Display the DataFrame
df_usage_metrics

Total costs: $0.0001


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,994,870,0,124,2


## Inspecting Results

In [71]:
flow.state["score_crews_results"][0].pydantic.personal_info.name

'João Moura'

In [72]:
import pandas as pd
from IPython.display import display, HTML

lead_scoring_result = flow.state["score_crews_results"][0].pydantic

# Create a dictionary with the nested structure flattened
data = {
    'Name': lead_scoring_result.personal_info.name,
    'Job Title': lead_scoring_result.personal_info.job_title,
    'Role Relevance': lead_scoring_result.personal_info.role_relevance,
    'Professional Background': lead_scoring_result.personal_info.professional_background,
    'Company Name': lead_scoring_result.company_info.company_name,
    'Industry': lead_scoring_result.company_info.industry,
    'Company Size': lead_scoring_result.company_info.company_size,
    'Revenue': lead_scoring_result.company_info.revenue,
    'Market Presence': lead_scoring_result.company_info.market_presence,
    'Lead Score': lead_scoring_result.lead_score.score,
    'Scoring Criteria': ', '.join(lead_scoring_result.lead_score.scoring_criteria),
    'Validation Notes': lead_scoring_result.lead_score.validation_notes
}

# Convert the dictionary to a DataFrame
df = pd.DataFrame.from_dict(data, orient='index', columns=['Value'])

# Reset the index to turn the original column names into a regular column
df = df.reset_index()

# Rename the index column to 'Attribute'
df = df.rename(columns={'index': 'Attribute'})

# Create HTML table with bold attributes and left-aligned values
html_table = df.style.set_properties(**{'text-align': 'left'}) \
                     .format({'Attribute': lambda x: f'<b>{x}</b>'}) \
                     .hide(axis='index') \
                     .to_html()

# Display the styled HTML table
display(HTML(html_table))

Attribute,Value
Name,João Moura
Job Title,Director of Engineering
Role Relevance,8
Professional Background,Experienced engineering leader with a background in technology and leadership roles.
Company Name,Clearbit
Industry,Internet Software & Services
Company Size,120
Revenue,None
Market Presence,0
Lead Score,85


In [73]:
import textwrap

result_text = emails[0].raw
wrapped_text = textwrap.fill(result_text, width=80)
print(wrapped_text)

Subject: Unlock Automation Potential with CrewAI  Hi there,  Are you ready to
revolutionize your automation processes? CrewAI is the ultimate platform for
orchestrating AI Agents tailored for enterprise companies seeking efficient
Agentic automation solutions.   Maximize your operational efficiency and
schedule a personalized demo today to witness the power of multi-agent
orchestration in action. Don't miss out on the opportunity to streamline your
operations effortlessly.  Schedule your demo now and embark on a journey towards
unparalleled automation excellence with CrewAI.  Best,   CrewAI Team


In [74]:
flow.state['score_crews_results'][0].model_dump()['tasks_output']

[{'description': "Collect and analyze the following information about the lead:\n- Personal Information:\n  - Name: Obtain the full name of the lead.\n  - Job Title: Determine the lead's current job title.\n  - Role Relevance: Assess how relevant the lead's role is to the decision-making process on a scale from 0 to 10.\n  - Professional Background: Optionally, gather a brief description of the lead's professional background.\n\n- Company Information:\n  - Company Name: Identify the name of the company the lead works for.\n  - Industry: Determine the industry in which the company operates.\n  - Company Size: Estimate the size of the company in terms of employee count.\n  - Revenue: If available, collect information on the annual revenue of the company.\n  - Market Presence: Evaluate the company's market presence on a scale from 0 to 10.\n\n- Our Company and Product:\n  - Company Name: CrewAI\n  - Product: Multi-Agent Orchestration Platform\n  - ICP: Enterprise companies looking into Ag

## How Complex Can it Get?

In [41]:
from crewai import Flow
from crewai.flow.flow import listen, start, and_, or_, router

class SalesPipeline(Flow):
    
  @start()
  def fetch_leads(self):
    # Pull our leads from the database
    # This is a mock, in a real-world scenario, this is where you would
    # fetch leads from a database
    leads = [
      {
        "lead_data": {
          "name": "João Moura",
          "job_title": "Director of Engineering",
          "company": "Clearbit",
          "email": "joao@clearbit.com",
          "use_case": "Using AI Agent to do better data enrichment."
        },
      },
    ]
    return leads

  @listen(fetch_leads)
  def score_leads(self, leads):
    scores = lead_scoring_crew.kickoff_for_each(leads)
    self.state["score_crews_results"] = scores
    return scores

  @listen(score_leads)
  def store_leads_score(self, scores):
    # Here we would store the scores in the database
    return scores

  @listen(score_leads)
  def filter_leads(self, scores):
    return [score for score in scores if score['lead_score'].score > 70]

  @listen(and_(filter_leads, store_leads_score))
  def log_leads(self, leads):
    print(f"Leads: {leads}")
    return leads

  @router(filter_leads)
  def count_leads(self, scores):
    if len(scores) > 10:
      return "high"
    elif len(scores) > 5:
      return "medium"
    else:
      return "low"

  @listen("high")
  def store_in_salesforce(self, leads):
    # Store high-value leads in Salesforce
    return leads

  @listen("medium")
  def send_to_sales_team(self, leads):
    # Send medium-value leads to the sales team
    return leads

  @listen("low")
  def write_email(self, leads):
    scored_leads = [lead.to_dict() for lead in leads]
    emails = email_writing_crew.kickoff_for_each(scored_leads)
    return emails

  @listen(write_email)
  def send_email(self, emails):
    # Here we would send the emails to the leads
    return emails


In [46]:
flow = SalesPipeline()
flow.plot()

╭──────────────────────────────────────────────── Flow Execution ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: SalesPipeline                                                                                            │
│  ID: 8ce145f4-cf1c-40d1-8049-d26485c1b4b6                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Plot saved as crewai_flow.html


In [47]:
from IPython.display import IFrame

IFrame(src='./crewai_flow.html', width='150%', height=600)